In [25]:
import anndata as ad
import pandas as pd
import numpy as np
import os
import pickle

# 1. Get Fingerprints

In [26]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [27]:
de_train = ad.read_h5ad('../data/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
de_test = ad.read_h5ad('../data/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')

In [28]:
de = ad.concat([de_train, de_test])
sm_smiles = de.obs[['sm_name', 'SMILES']].drop_duplicates()
sm_smiles['ECFP:2'] = smiles_to_fingerprints(sm_smiles['SMILES'])
sm_smiles = sm_smiles.rename(columns = {'SMILES': 'smiles', 'sm_name': 'perturbagen'})

In [29]:
sm_smiles

,perturbagen,smiles,ECFP:2
"NK cells, TIE2 Kinase Inhibitor",TIE2 Kinase Inhibitor,COc1ccc2cc(-c3[nH]c(-c4ccc([S+](C)[O-])cc4)nc3...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, MK-5108",MK-5108,O=C(O)[C@]1(Cc2cccc(Nc3nccs3)n2)CC[C@@H](Oc2cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Lapatinib",Lapatinib,CS(=O)(=O)CCNCc1ccc(-c2ccc3ncnc(Nc4ccc(OCc5ccc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, Belinostat",Belinostat,O=C(/C=C/c1cccc(S(=O)(=O)Nc2ccccc2)c1)NO,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"B cells, Dabrafenib",Dabrafenib,CC(C)(C)c1nc(-c2cccc(NS(=O)(=O)c3c(F)cccc3F)c2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...
"B cells, GLPG0634",GLPG0634,O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Mubritinib (TAK 165)",Mubritinib (TAK 165),FC(F)(F)c1ccc(/C=C/c2nc(COc3ccc(CCCCn4ccnn4)cc...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, Vanoxerine",Vanoxerine,Fc1ccc(C(OCCN2CCN(CCCc3ccccc3)CC2)c2ccc(F)cc2)cc1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
"NK cells, SB525334",SB525334,Cc1cccc(-c2[nH]c(C(C)(C)C)nc2-c2ccc3nccnc3c2)n1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


# 2. Load Pubchem

In [30]:
#PubChemCIDs were obtained with the Chem-PerturBridge pipeline with the PubChemPy package
df_emb_op3 = pd.read_csv('../files/df_pubchem_op3.csv')
df_emb_op3['pubchem_cid'] = df_emb_op3['pubchem_cid'].astype(str)

# 3. Get embeddings and JOIN

In [31]:
epoch_dirs = ['epoch_epoch_0019']

In [32]:
epoch_dirs

['epoch_epoch_0019']

In [33]:
import os
path = '../data/benchmark/resources/datasets/neurips-2023-data-subsample'
os.makedirs(path, exist_ok=True)

for epoch_dir in epoch_dirs:
    
    df_pert_all = pd.read_pickle(f'../files/single_run_all_25_epochs/{epoch_dir}/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_all', 
                                     'code': 'code_all', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_all'})
    

    
    df_emb_op3_merged = df_emb_op3.merge(df_pert_all, left_on='pubchem_cid', right_on='symbol_all', how='left')\
                            .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

    mask = df_emb_op3_merged['lpm_style_embeddings_all'].isna() | df_emb_op3_merged['ECFP:2'].isna()
    df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]
    
    tag = int(epoch_dir.split('_')[-1])
    
    df_emb_op3_all_to_save = df_emb_op3_merged_filtered.rename(columns={'lpm_style_embeddings_all': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
    df_emb_op3_all_to_save.to_pickle(f'{path}/op3_emb_all_{tag + 1}.pkl')

In [34]:
for epoch_dir in epoch_dirs:
    
    df_pert_l1000 = pd.read_pickle(f'../../lpm_style/files/single_run_l1000_25_epochs/{epoch_dir}/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_l1000', 
                                     'code': 'code_l1000', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_l1000'})
    

    
    df_emb_op3_merged = df_emb_op3.merge(df_pert_l1000, left_on='pubchem_cid', right_on='symbol_l1000', how='left')\
                            .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

    mask = df_emb_op3_merged['lpm_style_embeddings_l1000'].isna() | df_emb_op3_merged['ECFP:2'].isna()
    df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]
    
    tag = int(epoch_dir.split('_')[-1])
    
    df_emb_op3_l1000_to_save = df_emb_op3_merged_filtered.rename(columns={'lpm_style_embeddings_l1000': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
    df_emb_op3_l1000_to_save.to_pickle(f'{path}/op3_emb_l1000_{tag + 1}.pkl')

In [35]:
df_pert_all = pd.read_pickle(f'../../lpm_style/files/single_run_all_25_epochs/{epoch_dir}/df_pert.pkl')\
                    .rename(columns={'symbol': 'symbol_all', 
                                     'code': 'code_all', 
                                     'lpm_style_embeddings': 
                                     'lpm_style_embeddings_all'})

    
df_emb_op3_merged = df_emb_op3.merge(df_pert_all, left_on='pubchem_cid', right_on='symbol_all', how='left')\
                        .merge(sm_smiles, left_on='perturbagen', right_on='perturbagen',  how='left')

mask = df_emb_op3_merged['ECFP:2'].isna()
df_emb_op3_merged_filtered = df_emb_op3_merged[~mask]

tag = int(epoch_dir.split('_')[-1])

df_emb_op3_all_to_save = df_emb_op3_merged_filtered.rename(columns={'lpm_style_embeddings_all': 'LPM_emb'})[['perturbagen', 'LPM_emb', 'smiles', 'ECFP:2']].reset_index(drop=True)
df_emb_op3_all_to_save.to_pickle(f'{path}/op3_emb_fp.pkl')

# 4. Split

In [36]:
ratio = 0.25

In [37]:
op3_train_subsample = ad.read_h5ad('../data/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
op3_test_subsample = ad.read_h5ad('../data/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')
df = pd.read_csv('../data/benchmark/resources/datasets/neurips-2023-data/id_map.csv')

In [38]:
op3_test_subsample

AnnData object with n_obs × n_vars = 151 × 5317
    obs: 'sm_cell_type', 'cell_type', 'sm_name', 'sm_lincs_id', 'SMILES', 'split', 'control'
    uns: 'dataset_description', 'dataset_id', 'dataset_name', 'dataset_organism', 'dataset_reference', 'dataset_summary', 'dataset_url', 'single_cell_obs'
    layers: 'AveExpr', 'B', 'P.Value', 'adj.P.Value', 'clipped_sign_log10_pval', 'is_de', 'is_de_adj', 'logFC', 'sign_log10_adj_pval', 'sign_log10_pval', 't'

In [39]:
op3_subsample = ad.concat([op3_train_subsample, op3_test_subsample], uns_merge='same')

In [40]:
df_single_cell_obs = pd.concat([op3_train_subsample.uns['single_cell_obs'], op3_test_subsample.uns['single_cell_obs']])

In [41]:
compounds = np.array(op3_subsample.obs['sm_name'].unique())

In [42]:
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)

In [43]:
op3_subsample.obs['new_split'] = np.where(op3_subsample.obs['sm_name'].isin(test_sample), 'test', 'train')

In [44]:
op3_train_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'train'].copy()
op3_train_subsample_.uns['single_cell_obs'] = df_single_cell_obs[~df_single_cell_obs['sm_name'].isin(test_sample)]
op3_train_subsample_.write_h5ad('../data/benchmark/resources/datasets/neurips-2023-data-subsample/de_train.h5ad', compression='gzip')

In [45]:
op3_test_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'test'].copy()
op3_test_subsample_.uns['single_cell_obs'] = df_single_cell_obs[df_single_cell_obs['sm_name'].isin(test_sample)]
op3_test_subsample_.write_h5ad('../data/benchmark/resources/datasets/neurips-2023-data-subsample/de_test.h5ad', compression='gzip')

In [46]:
op3_test_subsample_.obs[['sm_name', 'cell_type']].reset_index(drop=True).reset_index().rename(columns={'index': 'id'}).to_csv('../data/benchmark/resources/datasets/neurips-2023-data-subsample/id_map.csv', index=False)

In [47]:
assert len(set(op3_test_subsample_.obs['sm_name']).intersection(set(op3_train_subsample_.obs['sm_name']))) == 0

In [48]:
len(op3_test_subsample_.obs['sm_name'].unique()) + len(op3_train_subsample_.obs['sm_name'].unique())

140